# ECMWF IFS High-Resolution Forecast

Real-time global weather forecasts from ECMWF's Integrated Forecasting System (IFS), updated 4 times daily at 00, 06, 12, and 18 UTC.

**Dataset Specifications:**
- **Resolution:** 0.25° (~25 km)
- **Coverage:** Global (90°N to 90°S, 0°E to 360°E)
- **Forecast Range:** 0-240 hours (00z/12z), 0-90 hours (06z/18z)
- **Temporal Resolution:** 3-hourly (0-144h), 6-hourly (150-240h)
- **Variables:** 9 essential parameters (temperature, wind, pressure, precipitation, radiation, water vapor)

**Data Source:** [ECMWF Open Data](https://www.ecmwf.int/en/forecasts/datasets/open-data)

**License:** CC-BY-4.0

In [ ]:
# Install dependencies (run once)
# !pip install xarray[complete]>=2025.1.2 zarr>=3.0.8 matplotlib cartopy numpy

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from datetime import datetime

## Load Dataset

Open the latest ECMWF IFS forecast from cloud-optimized Zarr storage:

In [ ]:
# Replace <username> with your GitHub username
ds = xr.open_zarr(
    "https://<username>.github.io/ecmwf_open_data_to_zarr/data/ecmwf/ifs/forecast-15-day/latest.zarr",
    consolidated=True,
    chunks=None
)

ds

## Dataset Structure

The dataset contains 9 variables with dimensions `(init_time, lead_time, latitude, longitude)`:

In [ ]:
# Display available variables
print("Available Variables:")
for var in ds.data_vars:
    print(f"  - {var}")

# Show forecast initialization time and range
print(f"\nForecast Initialization: {ds.init_time.values}")
print(f"Forecast Lead Times: {len(ds.lead_time)} steps")
print(f"  Range: 0 to {int(ds.lead_time.values[-1] / np.timedelta64(1, 'h'))} hours")

## Global Temperature Map

Visualize current conditions (0-hour forecast):

In [ ]:
# Convert temperature from Kelvin to Celsius
temp_celsius = ds["temperature_2m"] - 273.15

# Select initial forecast time (analysis)
temp_0h = temp_celsius.isel(lead_time=0)

# Create map
fig = plt.figure(figsize=(16, 8))
ax = plt.axes(projection=ccrs.Robinson())

# Plot temperature
temp_0h.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdYlBu_r",
    vmin=-40,
    vmax=40,
    cbar_kwargs={
        'label': 'Temperature (°C)',
        'shrink': 0.6,
        'pad': 0.05
    }
)

# Add features
ax.coastlines(linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.5)
ax.gridlines(draw_labels=False, alpha=0.3)

plt.title(
    f"ECMWF IFS 2m Temperature - Analysis Time\n"
    f"Init: {str(ds.init_time.values)[:16]} UTC",
    fontsize=14,
    pad=20
)

plt.tight_layout()
plt.show()

## 24-Hour Temperature Forecast

View the forecast 24 hours ahead:

In [ ]:
# Select 24-hour forecast
temp_24h = temp_celsius.sel(lead_time="1 days")

# Create map
fig = plt.figure(figsize=(16, 8))
ax = plt.axes(projection=ccrs.Robinson())

# Plot temperature
temp_24h.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdYlBu_r",
    vmin=-40,
    vmax=40,
    cbar_kwargs={
        'label': 'Temperature (°C)',
        'shrink': 0.6,
        'pad': 0.05
    }
)

ax.coastlines(linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.5)
ax.gridlines(draw_labels=False, alpha=0.3)

# Calculate valid time
valid_time = ds.init_time + temp_24h.lead_time
plt.title(
    f"ECMWF IFS 2m Temperature - 24h Forecast\n"
    f"Valid: {str(valid_time.values)[:16]} UTC",
    fontsize=14,
    pad=20
)

plt.tight_layout()
plt.show()

## Location-Specific Time Series

Extract forecast for a specific city:

In [ ]:
# Define locations
locations = {
    "London": {"lat": 51.5, "lon": -0.1},
    "New York": {"lat": 40.7, "lon": -74.0},
    "Tokyo": {"lat": 35.7, "lon": 139.7},
    "Sydney": {"lat": -33.9, "lon": 151.2}
}

# Create figure
fig, ax = plt.subplots(figsize=(14, 6))

# Plot temperature forecast for each location
for city, coords in locations.items():
    # Select nearest grid point
    city_data = ds.sel(
        latitude=coords["lat"],
        longitude=coords["lon"] % 360,  # Convert to 0-360 range
        method="nearest"
    )
    
    # Convert temperature to Celsius
    temp_c = city_data["temperature_2m"] - 273.15
    
    # Convert lead_time to hours for plotting
    hours = city_data.lead_time.values / np.timedelta64(1, 'h')
    
    # Plot
    ax.plot(hours, temp_c.values, marker='o', label=city, linewidth=2, markersize=4)

ax.set_xlabel("Forecast Lead Time (hours)", fontsize=12)
ax.set_ylabel("Temperature (°C)", fontsize=12)
ax.set_title(
    f"ECMWF IFS Temperature Forecast - City Comparison\n"
    f"Init: {str(ds.init_time.values)[:16]} UTC",
    fontsize=14,
    pad=15
)
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Wind Speed Analysis

Calculate and visualize 10-meter wind speed:

In [ ]:
# Calculate wind speed from components
wind_speed = np.sqrt(ds["wind_u_10m"]**2 + ds["wind_v_10m"]**2)

# Select 0-hour forecast
wind_0h = wind_speed.isel(lead_time=0)

# Create map
fig = plt.figure(figsize=(16, 8))
ax = plt.axes(projection=ccrs.Robinson())

# Plot wind speed
wind_0h.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="YlOrRd",
    vmin=0,
    vmax=25,
    cbar_kwargs={
        'label': 'Wind Speed (m/s)',
        'shrink': 0.6,
        'pad': 0.05
    }
)

ax.coastlines(linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.5)
ax.gridlines(draw_labels=False, alpha=0.3)

plt.title(
    f"ECMWF IFS 10m Wind Speed - Analysis Time\n"
    f"Init: {str(ds.init_time.values)[:16]} UTC",
    fontsize=14,
    pad=20
)

plt.tight_layout()
plt.show()

## Precipitation Accumulation

View accumulated precipitation over forecast period:

In [ ]:
# Convert precipitation from meters to millimeters
precip_mm = ds["total_precipitation"] * 1000

# Select 24-hour accumulation
precip_24h = precip_mm.sel(lead_time="1 days")

# Create map
fig = plt.figure(figsize=(16, 8))
ax = plt.axes(projection=ccrs.Robinson())

# Plot precipitation with logarithmic scale
precip_24h.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="Blues",
    vmin=0.1,
    vmax=100,
    norm=plt.matplotlib.colors.LogNorm(vmin=0.1, vmax=100),
    cbar_kwargs={
        'label': 'Accumulated Precipitation (mm)',
        'shrink': 0.6,
        'pad': 0.05
    }
)

ax.coastlines(linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.5)
ax.gridlines(draw_labels=False, alpha=0.3)

plt.title(
    f"ECMWF IFS 24h Accumulated Precipitation\n"
    f"Valid: {str((ds.init_time + precip_24h.lead_time).values)[:16]} UTC",
    fontsize=14,
    pad=20
)

plt.tight_layout()
plt.show()

## Regional Focus: North America

Extract and visualize data for a specific region:

In [ ]:
# Define North America bounding box
north_america = ds.sel(
    latitude=slice(60, 20),
    longitude=slice(235, 295)  # -125 to -65 in 0-360 range
)

# Calculate MSLP in hPa and temperature in Celsius
mslp_hpa = north_america["mean_sea_level_pressure"] / 100
temp_c = north_america["temperature_2m"] - 273.15

# Select 48-hour forecast
mslp_48h = mslp_hpa.sel(lead_time="2 days")
temp_48h = temp_c.sel(lead_time="2 days")

# Create map
fig = plt.figure(figsize=(14, 10))
ax = plt.axes(projection=ccrs.LambertConformal(
    central_longitude=-95,
    central_latitude=40
))

# Set extent
ax.set_extent([-125, -65, 20, 60], crs=ccrs.PlateCarree())

# Plot temperature
temp_48h.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdYlBu_r",
    vmin=-30,
    vmax=35,
    cbar_kwargs={
        'label': 'Temperature (°C)',
        'shrink': 0.7,
        'pad': 0.05
    }
)

# Add MSLP contours
contours = ax.contour(
    north_america.longitude,
    north_america.latitude,
    mslp_48h,
    transform=ccrs.PlateCarree(),
    levels=np.arange(960, 1040, 4),
    colors='black',
    linewidths=1,
    alpha=0.5
)
ax.clabel(contours, inline=True, fontsize=9, fmt='%d')

# Add features
ax.coastlines(linewidth=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3, alpha=0.5)
ax.gridlines(draw_labels=True, alpha=0.3)

valid_time = ds.init_time + mslp_48h.lead_time
plt.title(
    f"ECMWF IFS: Temperature (filled) & MSLP (contours)\n"
    f"48h Forecast - Valid: {str(valid_time.values)[:16]} UTC",
    fontsize=14,
    pad=15
)

plt.tight_layout()
plt.show()

## Multi-Variable Analysis

Compare multiple forecast variables for a location:

In [ ]:
# Select location (e.g., London)
location = ds.sel(latitude=51.5, longitude=-0.1, method="nearest")

# Convert lead_time to hours
hours = location.lead_time.values / np.timedelta64(1, 'h')

# Create subplots
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Temperature
temp_c = location["temperature_2m"] - 273.15
axes[0].plot(hours, temp_c.values, 'r-', linewidth=2)
axes[0].set_ylabel('Temperature (°C)', fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_title('Temperature Forecast', fontsize=12, pad=10)

# Wind Speed
wind_speed = np.sqrt(
    location["wind_u_10m"]**2 + location["wind_v_10m"]**2
)
axes[1].plot(hours, wind_speed.values, 'b-', linewidth=2)
axes[1].set_ylabel('Wind Speed (m/s)', fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_title('10m Wind Speed Forecast', fontsize=12, pad=10)

# Pressure
mslp_hpa = location["mean_sea_level_pressure"] / 100
axes[2].plot(hours, mslp_hpa.values, 'g-', linewidth=2)
axes[2].set_xlabel('Forecast Lead Time (hours)', fontsize=11)
axes[2].set_ylabel('MSLP (hPa)', fontsize=11)
axes[2].grid(True, alpha=0.3)
axes[2].set_title('Mean Sea Level Pressure Forecast', fontsize=12, pad=10)

fig.suptitle(
    f"ECMWF IFS Multi-Variable Forecast - London\n"
    f"Init: {str(ds.init_time.values)[:16]} UTC",
    fontsize=15,
    y=0.995
)

plt.tight_layout()
plt.show()

## Data Export

Extract and save data for offline analysis:

In [ ]:
# Select a region and convert to pandas DataFrame
location_forecast = ds.sel(
    latitude=40.7,
    longitude=-74.0,
    method="nearest"
).to_dataframe()

# Display first few rows
print("Forecast data as DataFrame:")
print(location_forecast.head())

# Save to CSV (optional)
# location_forecast.to_csv('ecmwf_forecast_nyc.csv')
# print("\nData saved to ecmwf_forecast_nyc.csv")

## Additional Resources

- **Dataset Documentation:** See README.md for full variable descriptions and technical details
- **ECMWF IFS Model:** [ECMWF Model Documentation](https://www.ecmwf.int/en/forecasts/documentation-and-support)
- **Xarray Tutorial:** [xarray.pydata.org/en/stable/tutorials-and-videos.html](https://xarray.pydata.org/en/stable/tutorials-and-videos.html)
- **GitHub Repository:** [github.com/\<username\>/ecmwf_open_data_to_zarr](https://github.com/<username>/ecmwf_open_data_to_zarr)

---

*This notebook demonstrates basic access patterns for ECMWF IFS forecast data. For more advanced analysis, explore xarray's powerful selection, computation, and visualization capabilities.*